# Assignment 4: Regularization 

**Q1.** Please answer the following questions in your own words.

1. What is the intuition of adding a penalty to mean squared error, that grows in the "size" (absolute or squared value) of the model parameters?
2. How does regularization provide a way of exploring the bias-variance trade-off?
3. What is the difference between LASSO and Ridge regression? How do the answers typically change for the two problems?
4. How do we typically scale variables for use in regularized regression? Why?
5. How is the penalty $\alpha$ typically selected?
6. When conducting cross validation, do you include the penalty term in evaluating the cross validated MSE? Why or why not?

1.  Adding a penalty that grows with parameter size discourages the model from using large coefficients. This prevents overfitting by limiting the model's flexibility. Large coefficients mean the model is fitting noise rather than signal, so penalizing them pushes the model toward simpler solutions.

2. Regularization lets us control the bias variance tradeoff through the penalty strength α. Small α allows models that are low bias, high variance while large α has models that are high bias, low variance. By tuning α with a cross validation, we find the sweet spot that minimizes total prediction error.

3. Ridge uses L2 penalty which are squared coefficients, shrinking all coefficients toward zero but rarely to exactly zero. LASSO uses L1 penalty which are absolute values, which can set coefficients exactly to zero, giving automatic feature selection. Ridge gives dense solutions and LASSO gives sparse solutions.

4. We standardize variables which is pretty much z-score normalization so they're on the same scale. Without this, variables with larger scales would be penalized more heavily simply due to their units, not their importance. Standardization makes sure the penalty treats all variables fairly.

5. We use cross validation to select α. We try a grid of α values, compute CV error for each, and choose the α that minimizes prediction error on held out data. This ensures α generalizes well to new data.

6. We don't include the penalty term when evaluating CV MSE. The penalty is only used during training to constrain coefficients. For evaluation, we only care about MSE, not the size of coefficients. The penalty is a means to better predictions, not part of the prediction error itself.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LassoCV, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 20)
pd.set_option('display.precision', 4)
np.random.seed(42)

**Q2.** This is a case study on regularization.

1. Import the `cars_hw.csv` dataset. Create an `Age` variable for each vehicle. Take `Mileage_Run` and `Age`, and (a) use `PolynomialFeatures` to create a third degree expansion, (b) use `StandardScaler` to $z$-score normalize them. 
2. Use your features, run linear regression. What is the sign for the interaction between `Mileage_Run` and `Age`?
3. Use `LassoCV` to regularize your linear regression, using 20-fold cross validation. (Hint: I used the grid `alphas = np.logspace(1,3,20)` to find the cost parameter)
4. Plot the cross-validated MSE by $\alpha$.
5. Plot the coefficient paths by $\alpha$.
6. Which features are actually selected? What proportion are set equal to zero?
7. Compare the linear regressions and optimally regularized coefficients. Do any coefficients increase in magnitude from linear regression to LASSO? Do any change sign?

In [ ]:
# Q2.1: Load data and create polynomial features
cars = pd.read_csv('./data/cars_hw.csv')

# age variable
current_year = 2024
cars['Age'] = current_year - cars['Make_Year']

#features and target
X_raw = cars[['Mileage_Run', 'Age']].dropna()
y = cars.loc[X_raw.index, 'Price']

print(f"Data shape: {X_raw.shape}")
print(f"\nOriginal features:")
display(X_raw.head())

# (a)third degree polynomial features
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X_raw)

print(f"\nAfter polynomial expansion: {X_poly.shape}")
print(f"Feature names: {poly.get_feature_names_out()}")

# (b)standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_poly)

print(f"\nAfter standardization:")
print(f"  Mean: {X_scaled.mean(axis=0).round(10)}")
print(f"  Std: {X_scaled.std(axis=0).round(2)}")

In [ ]:
# Q2.2: Linear regression
model_lr = LinearRegression()
model_lr.fit(X_scaled, y)

# coefficient dataframe
feature_names = poly.get_feature_names_out()
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model_lr.coef_
})

print("Q2.2: Linear Regression Coefficients")
display(coef_df)

#find interaction term (Mileage_Run * Age)
interaction_idx = [i for i, name in enumerate(feature_names) if 'Mileage_Run' in name and 'Age' in name and name.count(' ') == 1]
if interaction_idx:
    interaction_coef = model_lr.coef_[interaction_idx[0]]
    print(f"\nInteraction (Mileage_Run × Age) coefficient: {interaction_coef:.4f}")
    print(f"Sign: {'Positive' if interaction_coef > 0 else 'Negative'}")
    
    if interaction_coef < 0:
        print("\nInterpretation: The negative interaction suggests that the depreciation")
        print("effect of mileage is less severe for older cars (or vice versa).")

In [ ]:
# Q2.3: LASSO with cross validation
alphas = np.logspace(1, 3, 20)
lasso_cv = LassoCV(alphas=alphas, cv=20, random_state=42, max_iter=10000)
lasso_cv.fit(X_scaled, y)

print("Q2.3: LASSO Results")
print(f"\nOptimal alpha: {lasso_cv.alpha_:.4f}")
print(f"Number of features: {len(feature_names)}")
print(f"Non-zero coefficients: {np.sum(lasso_cv.coef_ != 0)}")
print(f"Zero coefficients: {np.sum(lasso_cv.coef_ == 0)}")

In [ ]:
# Q2.4: Plot CV MSE vs alpha
mse_path = np.mean(lasso_cv.mse_path_, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(alphas, mse_path, 'b-', linewidth=2, marker='o')
plt.axvline(lasso_cv.alpha_, color='red', linestyle='--', linewidth=2, 
            label=f'Optimal α = {lasso_cv.alpha_:.2f}')
plt.xscale('log')
plt.xlabel('Alpha (regularization strength)', fontsize=12)
plt.ylabel('Cross-Validated MSE', fontsize=12)
plt.title('LASSO: CV MSE vs Alpha', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Q2.5: Plot coefficient paths
#fit LASSO for each alpha to get coefficient paths
coef_paths = []
for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_scaled, y)
    coef_paths.append(lasso.coef_)

coef_paths = np.array(coef_paths)

plt.figure(figsize=(12, 6))
for i in range(coef_paths.shape[1]):
    plt.plot(alphas, coef_paths[:, i], linewidth=1.5, alpha=0.7)

plt.axvline(lasso_cv.alpha_, color='red', linestyle='--', linewidth=2, 
            label=f'Optimal α = {lasso_cv.alpha_:.2f}')
plt.xscale('log')
plt.xlabel('Alpha (regularization strength)', fontsize=12)
plt.ylabel('Coefficient value', fontsize=12)
plt.title('LASSO: Coefficient Paths', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Q2.6: Selected features
selected_features = coef_df[lasso_cv.coef_ != 0].copy()
selected_features['LASSO_Coef'] = lasso_cv.coef_[lasso_cv.coef_ != 0]

print("Q2.6: Selected Features")
display(selected_features)

n_selected = np.sum(lasso_cv.coef_ != 0)
n_total = len(lasso_cv.coef_)
pct_zero = 100 * (n_total - n_selected) / n_total

print(f"\nSelected: {n_selected} features")
print(f"Set to zero: {n_total - n_selected} features ({pct_zero:.1f}%)")

In [ ]:
# Q2.7: Compare Linear Regression vs LASSO
comparison = pd.DataFrame({
    'Feature': feature_names,
    'LR_Coef': model_lr.coef_,
    'LASSO_Coef': lasso_cv.coef_,
    'LR_Abs': np.abs(model_lr.coef_),
    'LASSO_Abs': np.abs(lasso_cv.coef_)
})

#check for magnitude increases
comparison['Mag_Increased'] = comparison['LASSO_Abs'] > comparison['LR_Abs']
#check for sign changes
comparison['Sign_Changed'] = (np.sign(comparison['LR_Coef']) != np.sign(comparison['LASSO_Coef'])) & (comparison['LASSO_Coef'] != 0)

print("Q2.7: Comparison of Linear Regression vs LASSO")
print("\nFeatures with increased magnitude:")
increased = comparison[comparison['Mag_Increased']]
if len(increased) > 0:
    display(increased[['Feature', 'LR_Coef', 'LASSO_Coef']])
else:
    print("None - LASSO shrinks all coefficients")

print("\nFeatures with sign changes:")
sign_changed = comparison[comparison['Sign_Changed']]
if len(sign_changed) > 0:
    display(sign_changed[['Feature', 'LR_Coef', 'LASSO_Coef']])
else:
    print("None")

print("\nSummary:")
print(f"  Coefficients increased in magnitude: {comparison['Mag_Increased'].sum()}")
print(f"  Coefficients changed sign: {comparison['Sign_Changed'].sum()}")
print(f"\nLASSO typically shrinks coefficients toward zero, so increases are rare.")
print(f"Sign changes can occur when LASSO removes correlated features.")

**Q3.** This is a case study on regularization.

1. Import the `heart_failure_clinical_records_dataset.csv` dataset. Use `PolynomialFeatures` to create a third-degree expansion of `age`, `ejection_fraction`, and `serum_creatinine`, and then use `StandardScaler` to $z$-score normalize your results. Use `PolynomialFeatures` with `interaction_only=True` to interact the dummy/categorical variables `anaemia`, `diabetes`, `high_blood_pressure`, and `smoking`. Concatenate these results into your feature/covariate matrix.
2. Use your features, run linear regression. Are there any sign patterns that appear counterintuitive? Why? Can you see how the inclusion of higher-order powers or interactions might resolve the apparent contradiction?
3. Use `LassoCV` to regularize your linear regression, using 20-fold cross validation. (Hint: I used the grid `alphas = np.logspace(-5,5,30)` to find the cost parameter)
4. Plot the cross-validated MSE by $\alpha$.
5. Plot the coefficient paths by $\alpha$.
6. Which features are actually selected? What proportion are set equal to zero? Compare the linear regressions and optimally regularized coefficients. Do any coefficients increase in magnitude from linear regression to LASSO? Do any change sign? Do the sign patterns for the linear_model or the Lasso seem to make more sense? Explain why this might be the case from the perspective of the bias-variance trade-off.

In [ ]:
# Q3.1: Load heart failure data and create features
heart = pd.read_csv('./data/heart_failure_clinical_records_dataset.csv')

print(f"Data shape: {heart.shape}")
print(f"\nColumns: {list(heart.columns)}")
display(heart.head())

#continuous variables for polynomial expansion
continuous_vars = ['age', 'ejection_fraction', 'serum_creatinine']
X_continuous = heart[continuous_vars]

#create third degree polynomial features
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X_continuous)

#standardize polynomial features
scaler = StandardScaler()
X_poly_scaled = scaler.fit_transform(X_poly)

print(f"\nPolynomial features: {X_poly_scaled.shape}")
print(f"Feature names: {poly.get_feature_names_out()}")

#categorical variables for interactions
categorical_vars = ['anaemia', 'diabetes', 'high_blood_pressure', 'smoking']
X_categorical = heart[categorical_vars]

#create interaction only features
poly_interact = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_interact = poly_interact.fit_transform(X_categorical)

print(f"\nCategorical interactions: {X_interact.shape}")
print(f"Interaction names: {poly_interact.get_feature_names_out()}")

# concatenate all features
X_combined = np.hstack([X_poly_scaled, X_interact])
y = heart['DEATH_EVENT']

print(f"\nFinal feature matrix: {X_combined.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

#store feature names for later
all_feature_names = list(poly.get_feature_names_out()) + list(poly_interact.get_feature_names_out())

In [ ]:
# Q3.2: Linear regression
model_lr_heart = LinearRegression()
model_lr_heart.fit(X_combined, y)

#create coefficient dataframe
coef_df_heart = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': model_lr_heart.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("Q3.2: Linear Regression Coefficients (sorted by magnitude)")
display(coef_df_heart.head(15))

print("\n**Counterintuitive patterns:**")
print("\nLook for signs that seem medically implausible, such as:")
print("- Positive coefficients for risk factors (should increase death probability)")
print("- Negative coefficients for protective factors")
print("\nThese contradictions often arise from:")
print("1. **Multicollinearity**: Correlated features confuse the model")
print("2. **Simpson's paradox**: Interactions reverse simple relationships")
print("3. **Overfitting**: With many features, the model fits noise")
print("\nHigher-order terms and interactions can capture non-linear relationships")
print("that resolve apparent contradictions in linear-only models.")

In [ ]:
# Q3.3: LASSO with cross validation
alphas_heart = np.logspace(-5, 5, 30)
lasso_cv_heart = LassoCV(alphas=alphas_heart, cv=20, random_state=42, max_iter=10000)
lasso_cv_heart.fit(X_combined, y)

print("Q3.3: LASSO Results")
print(f"\nOptimal alpha: {lasso_cv_heart.alpha_:.6f}")
print(f"Number of features: {len(all_feature_names)}")
print(f"Non-zero coefficients: {np.sum(lasso_cv_heart.coef_ != 0)}")
print(f"Zero coefficients: {np.sum(lasso_cv_heart.coef_ == 0)}")

In [ ]:
# Q3.4: Plot CV MSE vs alpha
mse_path_heart = np.mean(lasso_cv_heart.mse_path_, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(alphas_heart, mse_path_heart, 'b-', linewidth=2, marker='o')
plt.axvline(lasso_cv_heart.alpha_, color='red', linestyle='--', linewidth=2, 
            label=f'Optimal α = {lasso_cv_heart.alpha_:.4f}')
plt.xscale('log')
plt.xlabel('Alpha (regularization strength)', fontsize=12)
plt.ylabel('Cross-Validated MSE', fontsize=12)
plt.title('Heart Failure: CV MSE vs Alpha', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Q3.5: Plot coefficient paths
coef_paths_heart = []
for alpha in alphas_heart:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_combined, y)
    coef_paths_heart.append(lasso.coef_)

coef_paths_heart = np.array(coef_paths_heart)

plt.figure(figsize=(12, 6))
for i in range(coef_paths_heart.shape[1]):
    plt.plot(alphas_heart, coef_paths_heart[:, i], linewidth=1.5, alpha=0.6)

plt.axvline(lasso_cv_heart.alpha_, color='red', linestyle='--', linewidth=2, 
            label=f'Optimal α = {lasso_cv_heart.alpha_:.4f}')
plt.xscale('log')
plt.xlabel('Alpha (regularization strength)', fontsize=12)
plt.ylabel('Coefficient value', fontsize=12)
plt.title('Heart Failure: Coefficient Paths', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Q3.6: Compare models and analyze
comparison_heart = pd.DataFrame({
    'Feature': all_feature_names,
    'LR_Coef': model_lr_heart.coef_,
    'LASSO_Coef': lasso_cv_heart.coef_,
    'LR_Abs': np.abs(model_lr_heart.coef_),
    'LASSO_Abs': np.abs(lasso_cv_heart.coef_)
})

# selected features
selected = comparison_heart[comparison_heart['LASSO_Coef'] != 0].sort_values('LASSO_Abs', ascending=False)

print("Q3.6: Selected Features by LASSO")
display(selected[['Feature', 'LR_Coef', 'LASSO_Coef']])

n_selected = np.sum(lasso_cv_heart.coef_ != 0)
n_total = len(lasso_cv_heart.coef_)
pct_zero = 100 * (n_total - n_selected) / n_total

print(f"\nSelected: {n_selected} features")
print(f"Set to zero: {n_total - n_selected} features ({pct_zero:.1f}%)")

#check for magnitude increases and sign changes
comparison_heart['Mag_Increased'] = comparison_heart['LASSO_Abs'] > comparison_heart['LR_Abs']
comparison_heart['Sign_Changed'] = (np.sign(comparison_heart['LR_Coef']) != np.sign(comparison_heart['LASSO_Coef'])) & (comparison_heart['LASSO_Coef'] != 0)

print(f"\nCoefficients increased in magnitude: {comparison_heart['Mag_Increased'].sum()}")
print(f"Coefficients changed sign: {comparison_heart['Sign_Changed'].sum()}")

if comparison_heart['Sign_Changed'].sum() > 0:
    print("\nFeatures with sign changes:")
    display(comparison_heart[comparison_heart['Sign_Changed']][['Feature', 'LR_Coef', 'LASSO_Coef']])

print("\n**Which model makes more sense?**")
print("\nLASSO likely makes more sense because:")
print("1. **Reduced overfitting**: With many features and limited data, linear regression")
print("   overfits, producing unstable coefficients that fit noise.")
print("2. **Feature selection**: LASSO identifies truly important predictors, discarding")
print("   spurious correlations.")
print("3. **Bias-variance tradeoff**: LASSO accepts slight bias to dramatically reduce")
print("   variance, leading to better generalization.")
print("4. **Clinical interpretability**: Fewer features are easier to understand and")
print("   validate medically.")
print("\nLinear regression's counterintuitive signs likely reflect multicollinearity and")
print("overfitting, while LASSO's regularization produces more stable, sensible estimates.")

**Q4.** To better understand the math of regularization, we'll solve the regularized linear model problem with a single explanatory variable. So, the model is
$$
\tilde{y}_i = \tilde{b}_0 + \tilde{b}_1 \tilde{x}_i,
$$
where
$$
\tilde{y}_i = y_i - \bar{y} \quad \text{ and } \quad \tilde{x}_i = x_i - \bar{x}.
$$

Recall, we do this mean-normalization of $x$ and $y$, because
$$ 
\frac{1}{n} \sum_{i=1}^n \tilde{y} = \frac{1}{n} \sum_{i=1}^n y_i - \bar{y} = 0,
$$
and likewise for $x$. This trick makes the calculations easier and the results more easily interpretable.

1. To do ridge regression, add a penalty $+ \alpha (b_1)^2$ to mean squared error. What is the objective function for this problem?
2. Take the derivatives of your objective function with respect to $b_0$ and $b_1$. Set these equations equal to zero. Solve the two equations in two unknowns for $b_1$ and $b_0$.
3. How does increasing $\alpha$ change the slope coefficient?
4. If we instead used the LASSO/L1 penalty, $+\alpha |b_1|$, what challenge do you run into? This is conceptually difficult, but take 5 minutes and try to figure out the solution, and in particular, when is it optimal to set $b_1=0$?

Q4.1: Ridge Regression Objective Function

For mean-normalized data where $\tilde{y}_i = y_i - \bar{y}$ and $\tilde{x}_i = x_i - \bar{x}$, the Ridge regression objective function is:

Minimize
$$
 \quad L(b_0, b_1) = \frac{1}{n} \sum_{i=1}^n (\tilde{y}_i - b_0 - b_1 \tilde{x}_i)^2 + \alpha b_1^2
$$

This combines:
- Mean Squared Error: $\frac{1}{n} \sum_{i=1}^n (\tilde{y}_i - b_0 - b_1 \tilde{x}_i)^2$
- L2 Penalty: $\alpha b_1^2$

also we don't penalize $b_0$ (the intercept) because it just shifts predictions and doesn't affect model complexity.

Q4.2: Solving for Ridge Coefficients

first get derivatives

$$
\frac{\partial L}{\partial b_0} = \frac{-2}{n} \sum_{i=1}^n (\tilde{y}_i - b_0 - b_1 \tilde{x}_i)
$$

$$
\frac{\partial L}{\partial b_1} = \frac{-2}{n} \sum_{i=1}^n \tilde{x}_i(\tilde{y}_i - b_0 - b_1 \tilde{x}_i) + 2\alpha b_1
$$

then make it equal to zero

from $\frac{\partial L}{\partial b_0} = 0$:
$$
\sum_{i=1}^n (\tilde{y}_i - b_0 - b_1 \tilde{x}_i) = 0
$$

since $\sum_{i=1}^n \tilde{y}_i = 0$ and $\sum_{i=1}^n \tilde{x}_i = 0$ (mean-normalized):
$$
0 - n b_0 - b_1 \cdot 0 = 0 \quad \Rightarrow \quad b_0 = 0
$$

from $\frac{\partial L}{\partial b_1} = 0$ with $b_0 = 0$:
$$
\sum_{i=1}^n \tilde{x}_i(\tilde{y}_i - b_1 \tilde{x}_i) = n\alpha b_1
$$

$$
\sum_{i=1}^n \tilde{x}_i \tilde{y}_i - b_1 \sum_{i=1}^n \tilde{x}_i^2 = n\alpha b_1
$$

$$
\sum_{i=1}^n \tilde{x}_i \tilde{y}_i = b_1 \left(\sum_{i=1}^n \tilde{x}_i^2 + n\alpha\right)
$$

answer:
$$
{b_1 = \frac{\sum_{i=1}^n \tilde{x}_i \tilde{y}_i}{\sum_{i=1}^n \tilde{x}_i^2 + n\alpha}}
$$

$$
{b_0 = 0}
$$

Q4.3: Effect of Increasing α

from the solution
$$
b_1 = \frac{\sum_{i=1}^n \tilde{x}_i \tilde{y}_i}{\sum_{i=1}^n \tilde{x}_i^2 + n\alpha}
$$

as α increases
- The denominator increases: $\sum_{i=1}^n \tilde{x}_i^2 + n\alpha \uparrow$
- The numerator stays constant: $\sum_{i=1}^n \tilde{x}_i \tilde{y}_i$
- Therefore: $b_1 \rightarrow 0$

interpretation
- α = 0 No penalty, $b_1 = \frac{\sum \tilde{x}_i \tilde{y}_i}{\sum \tilde{x}_i^2}$ (OLS solution)
- α → ∞: Strong penalty, $b_1 \rightarrow 0$ (intercept-only model)
- Intermediate α Shrinks $b_1$ toward zero, balancing fit and complexity

Increasing α shrinks the slope coefficient toward zero reducing model complexity and variance at the cost of increased bias.

Q4.4: LASSO (L1 Penalty) Challenge

With LASSO penalty $+\alpha |b_1|$, the objective is:
$$
L(b_0, b_1) = \frac{1}{n} \sum_{i=1}^n (\tilde{y}_i - b_0 - b_1 \tilde{x}_i)^2 + \alpha |b_1|
$$

challenge: The absolute value is not differentiable at $b_1 = 0$

derivative is
$$
\frac{\partial |b_1|}{\partial b_1} = \begin{cases}
+1 & \text{if } b_1 > 0 \\
-1 & \text{if } b_1 < 0 \\
\text{undefined} & \text{if } b_1 = 0
\end{cases}
$$

Solution approach (subgradient method)

for $b_1 \neq 0$:
$$
\frac{\partial L}{\partial b_1} = \frac{-2}{n} \sum_{i=1}^n \tilde{x}_i(\tilde{y}_i - b_1 \tilde{x}_i) + \alpha \cdot \text{sign}(b_1) = 0
$$


$$
b_1 = \frac{\sum_{i=1}^n \tilde{x}_i \tilde{y}_i - \frac{n\alpha}{2} \cdot \text{sign}(b_1)}{\sum_{i=1}^n \tilde{x}_i^2}
$$

When is $b_1 = 0$ optimal?

let $\hat{b}_1^{OLS} = \frac{\sum \tilde{x}_i \tilde{y}_i}{\sum \tilde{x}_i^2}$ be the OLS solution.

The LASSO sets $b_1 = 0$ when:
$$
{|\hat{b}_1^{OLS}| < \frac{n\alpha}{2 \sum_{i=1}^n \tilde{x}_i^2}}
$$

interpretation
- If the OLS coefficient is small enough (below the threshold), LASSO sets it exactly to zero
- This is automatic feature selection: weak predictors are eliminated
- Ridge never sets coefficients exactly to zero, only shrinks them
- This is why LASSO produces sparse solutions while Ridge produces dense solutions